In [13]:
#1. 라이브러리 설치
def read_library():

    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import statsmodels.formula.api as smf
    import statsmodels.api as sm
    import platform
    import ast

    pd.set_option("display.float_format", "{:.2f}".format)

    if platform.system() == "Darwin":
        plt.rcParams["font.family"] = "AppleGothic"
    elif platform.system() == "Windows":
        plt.rcParams["font.family"] = "Malgun Gothic"
    else:
        plt.rcParams["font.family"] = "NanumGothic"

    return pd, np, plt, sns, stats, smf, sm, ast

#2. 데이터 로드

def load_data(pd):

    df = pd.read_csv("../원본 데이터 셋/2025_Airbnb_NYC_listings.csv")

    print("데이터 크기:", df.shape)
    print(df.head())

    return df

#3. 데이터 확인
def overview(df):

    print(df.info())
    print(df.describe())


#4. 가짜 결측 확인
def fake_null(df):

    fake_null = ['NaN','None','none','NULL','null','N/A','na','']

    fake_counts = df.isin(fake_null).sum()

    print(fake_counts[fake_counts > 0])


#5. Datetime 변환
def datetime(df, pd):

    date_cols = [
        "host_since",
        "first_review",
        "last_review",
        "calendar_last_scraped"
    ]

    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")

    return df


#6. 가격 컬럼 전처리
def price_clean(df, np):

    df["price"] = (
        df["price"]
        .astype(str)
        .str.replace("$","",regex=False)
        .str.replace(",","",regex=False)
        .astype(float)
    )

    df["price_log"] = np.log1p(df["price"])

    return df


#7. 리뷰관련 컬럼 결측치 처리
def missing(df):

    review_cols = [
        "number_of_reviews",
        "number_of_reviews_ltm",
        "number_of_reviews_l30d",
        "reviews_per_month",
        "estimated_occupancy_l365d"
    ]

    df[review_cols] = df[review_cols].fillna(0)

    df["review_scores_rating"] = df["review_scores_rating"].fillna(
        df["review_scores_rating"].median()
    )

    df["bathrooms"] = df["bathrooms"].fillna(0)
    df["bedrooms"] = df["bedrooms"].fillna(0)
    df["beds"] = df["beds"].fillna(0)

    return df


#8. 파생컬럼 생성
def feature_engineering(df, np):

    df["room_capacity"] = (
        df["bathrooms"] +
        df["bedrooms"] +
        df["beds"]
    )

    df["occupancy_rate"] = (
        df["estimated_occupancy_l365d"] / 365
    )

    df["log_reviews"] = np.log1p(df["number_of_reviews"])

    return df


# 9. Amenities Count(편의시설 길이 세는 함수)
def amenity_count(x):
    try:
        return len(ast.literal_eval(x))
    except:
        return 0

df["amenities_len"] = df["amenities"].apply(amenity_count)


#10. 룸타입 맵핑
def step10_regulation(df):

    property_map = {

        "Room in hotel": "Lodging",
        "Room in boutique hotel": "Lodging",

        "Entire rental unit": "Residential",
        "Private room in rental unit": "Residential",
        "Private room in home": "Residential",
        "Entire home": "Residential",
        "Entire condo": "Residential"
    }

    df["property_regulation_type"] = (
        df["property_type"]
        .map(property_map)
        .fillna("Other")
    )

    return df



# 11. 규제 범위 나누기
def step11_legal_filter(df):

    df["legal_flag"] = "Legal"

    df.loc[
        (df["property_regulation_type"]=="Residential") &
        (df["room_type"]=="Entire home/apt") &
        (df["minimum_nights"] < 30),
        "legal_flag"
    ] = "Illegal"

    df = df[
        ((df['property_regulation_type'] == 'Residential') & (df['legal_flag'] == 'Legal'))
        | (df['property_regulation_type'] == 'Lodging')
    ]

    return df


# 12. Strategy Classification
def strategy(row):

    if row["property_regulation_type"]=="Residential":

        if row["minimum_nights"] < 30:
            return "Residential_short_term"
        else:
            return "Residential_long_term"

    else:
        return "Hotel"

df["rental_strategy"] = df.apply(strategy, axis=1)


# 13. 매출 공식 검증
def revenue_verification(df):

    df["calc_revenue"] = (
        df["price"] *
        df["estimated_occupancy_l365d"]
    )

    print(
        "Revenue match:",
        (df["calc_revenue"]==df["estimated_revenue_l365d"]).mean()
    )

    return df


# 14. Outlier 제거해보고 확인
def outlier_check(df):

    df_example = df[df["price"] < 1000].copy()

    return df_example


# 15. Price Distribution
def price_distribution(df_example, sns, plt):

    sns.histplot(df_example["price"], bins=50)
    plt.title("Price Distribution")
    plt.show()


    sns.countplot(data=df_example, x="neighbourhood_group_cleansed")
    plt.title("Listing Count by Borough")
    plt.show()


# 16. Demand Analysis
def demand_analysis(df, sns, plt):

    sns.scatterplot(
        data=df,
        x="price",
        y="occupancy_rate",
        alpha=0.4
    )
    plt.title("Price vs Occupancy")
    plt.show()


    sns.scatterplot(
        data=df,
        x="number_of_reviews",
        y="occupancy_rate"
    )
    plt.title("Reviews vs Occupancy")
    plt.show()


# 17. Availability Analysis
def availability_analysis(df, sns, plt):

    sns.boxplot(
        data=df,
        x="availability_365",
        y="price"
    )
    plt.title("Availability vs Price")
    plt.show()


# 18. Price Quantile Strategy
def price_quantile(df, pd):

    df["price_q"] = pd.qcut(df["price"],4)

    print(
    df.groupby("price_q")[
    ["occupancy_rate","estimated_revenue_l365d"]
    ].median()
    )

    return df


# 19. Revenue 0 Listing
def zero_revenue(df):

    zero_ratio = (
        df["estimated_revenue_l365d"]==0
    ).mean()

    print("Revenue zero ratio:",zero_ratio)


# 20. Spearman Correlation
def spearman(df, stats):

    corr, p = stats.spearmanr(
        df["price"],
        df["occupancy_rate"]
    )

    print("Spearman:", corr)
    print("p:", p)


# 21. Kruskal Test
def kruskal(df, stats):

    groups = [

    df[df["rental_strategy"]=="Residential_long_term"]
    ["estimated_revenue_l365d"],

    df[df["rental_strategy"]=="Residential_short_term"]
    ["estimated_revenue_l365d"],

    df[df["rental_strategy"]=="Hotel"]
    ["estimated_revenue_l365d"]

    ]

    print(stats.kruskal(*groups))


# 22. ANOVA
def anova(df, smf, sm):

    anova_model = smf.ols(

    "estimated_revenue_l365d ~ C(rental_strategy)",

    data=df

    ).fit()

    print(sm.stats.anova_lm(anova_model))


# 23. Regression
def regression(df, smf):

    model = smf.ols(

    """
    occupancy_rate ~

    price_log
    + log_reviews
    + review_scores_rating
    + accommodates
    + minimum_nights
    + amenities_len
    """

    ,data=df

    ).fit()

    print(model.summary())

    return model


# 24. Regression Visualization
def regression_viz(model, plt):

    coef = model.params.drop("Intercept")

    coef.sort_values().plot(
    kind="barh",
    figsize=(8,6)
    )

    plt.title("Occupancy Drivers")
    plt.show()


# 25. Market Heatmap
def market_heatmap(df, sns, plt):

    heat = df.pivot_table(

    values="estimated_revenue_l365d",
    index="neighbourhood_group_cleansed",
    columns="room_type",
    aggfunc="median"

    )

    sns.heatmap(
    heat,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd"
    )

    plt.title("Revenue Heatmap")
    plt.show()


# 26. Investment Simulator
def recommend_investor(df, persona, initial_investment, top_n=10):

    # 1. 조건 필터링 (rental_strategy 사용)
    filtered = df[
        (df['rental_strategy'] == persona['rental_strategy']) &
        (df['neighbourhood_group_cleansed'] == persona['neighbourhood_group_cleansed']) &
        (df['room_type'] == persona['room_type'])
    ]

    if filtered.empty:
        print("조건에 맞는 데이터가 없습니다. 조건을 다시 입력하세요.")
        return pd.DataFrame()

    # 2. 세그먼트별 성과 지표 계산
    segment_summary = filtered.groupby(['room_type', 'accommodates']).agg(
        median_price=('price', 'median'),
        median_occupancy_rate=('occupancy_rate', 'median'),
        median_revenue=('estimated_revenue_l365d', 'median'),
        listing_count=('price', 'count'),
        median_reviews=('number_of_reviews', 'median')
    ).reset_index()

    # 수익성 지수
    segment_summary['profitability_index'] = (
        segment_summary['median_revenue'] / segment_summary['listing_count']
    )

    # 3. 블루오션/레드오션 판단
    listing_median = segment_summary['listing_count'].median()
    revenue_median = segment_summary['median_revenue'].median()

    def ocean(row):
        if row['listing_count'] >= listing_median and row['median_revenue'] >= revenue_median:
            return '레드오션'
        elif row['listing_count'] < listing_median and row['median_revenue'] >= revenue_median:
            return '블루오션'
        else:
            return '기타'

    segment_summary['시장포지션'] = segment_summary.apply(ocean, axis=1)

    # 4. 수익 계산
    segment_summary['월 매출'] = (
        segment_summary['median_price'] *
        segment_summary['median_occupancy_rate'] * 30
    )

    segment_summary['관리/유지비'] = segment_summary['월 매출'] * 0.25
    segment_summary['월 이익'] = segment_summary['월 매출'] - segment_summary['관리/유지비']

    segment_summary['회수기간(월)'] = np.where(
        segment_summary['월 이익'] > 0,
        initial_investment / segment_summary['월 이익'],
        np.nan
    )

    # 5. TOP 추천
    top_segments = segment_summary.sort_values(
        by='월 이익',
        ascending=False
    ).head(top_n)

    return top_segments


# Persona 입력
def create_persona():

    print("\n=== 투자자 페르소나 입력 ===")

    rental_strategy = input(
        "운영 전략 입력 (Residential_short_term / Residential_long_term / Hotel): "
    )

    room_type = input(
        "숙소 타입 입력 (Entire home/apt / Private room / Hotel room): "
    )

    neighbourhood_group_cleansed = input(
        "동네 입력 (Manhattan / Brooklyn / Queens / Bronx / Staten Island): "
    )

    persona = {
        'rental_strategy': rental_strategy,
        'room_type': room_type,
        'neighbourhood_group_cleansed': neighbourhood_group_cleansed
    }

    return persona


# 실행
persona = create_persona()

initial_investment = int(input("초기 투자금 입력 ($): "))

result = recommend_investor(df, persona, initial_investment, top_n=10)
result


=== 투자자 페르소나 입력 ===


,room_type,accommodates,median_price,median_occupancy_rate,median_revenue,listing_count,median_reviews,profitability_index,시장포지션,월 매출,관리/유지비,월 이익,회수기간(월)
6,Private room,7,302.00,0.70,57375.00,3,34.00,19125.00,블루오션,6329.59,1582.40,4747.19,3.16
7,Private room,8,328.00,0.63,59570.00,3,72.00,19856.67,블루오션,6200.55,1550.14,4650.41,3.23
4,Private room,5,290.50,0.51,53618.00,2,32.50,26809.00,블루오션,4429.13,1107.28,3321.85,4.52
3,Private room,4,132.00,0.70,29835.00,27,38.00,1105.00,기타,2766.58,691.64,2074.93,7.23
5,Private room,6,250.00,0.32,38760.00,5,9.00,7752.00,블루오션,2363.01,590.75,1772.26,8.46
1,Private room,2,96.00,0.70,22185.00,295,59.00,75.20,기타,2012.05,503.01,1509.04,9.94
2,Private room,3,90.00,0.70,22950.00,11,100.00,2086.36,기타,1886.30,471.58,1414.73,10.60
0,Private room,1,74.50,0.70,16830.00,54,56.50,311.67,기타,1561.44,390.36,1171.08,12.81


# 예시 시나리오
| 항목 | 시나리오 1: 브루클린 | 시나리오 2: 맨해튼 | 시나리오 3: 퀸즈 (최적안) |
|-----|------------------|------------------|------------------|
| 페르소나 | 단체 손님 전문 펜션지기 | 리스크 관리 전략가 | 3개월 컷 프로 부업러 |
| 투자 원금 | $25,000 (약 3,300만 원) | $30,000 (약 4,000만 원) | $15,000 (약 2,000만 원) |
| 숙소 유형 | Entire home (독채 전체) | Private room (개인실) | Private room (개인실) |
| 핵심 타겟 | 14인 이상 대규모 단체 | 5인 가족 및 비즈니스 팀 | 6인 우정 여행 / 소가족 |
| 평균 가격 | $374.00 | $385.50 | $269.00 |
| 시장 포지션 | 초블루오션 (숙소 5개뿐) | 블루오션 (숙소 4개뿐) | 블루오션 (숙소 8개뿐) |
| 월 순이익 | $2,766.58 | $3,029.87 | $4,228.46 |
| 회수 기간 | 9.04개월 | 9.9개월 | **3.55개월 (압도적)** |
| 한 줄 전략 | "레드오션 독채들 사이에서 '대형 인원'으로만 승부한다." | "비싼 맨해튼에서 1인실 버리고 5인실로 단가 싸움한다." | "가장 적은 돈 투자해서 가장 빨리 뽑고 돈 복사 시작한다." |